> 在线讲义：[第 10 章 · 保存与加载：存清单、不存对象，断点要存全](https://wikieden.github.io/pytorch-inside-out/lessons/10-save-load.html) · 本书《PyTorch 从里到外》

# 第 10 章 · 保存与加载：存清单、不存对象，断点要存全

本章回答：模型应该怎么存？为什么不建议 `torch.save(model)`？`weights_only` 防的是什么？断点续训除了参数还要存什么，一共多大？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/10-save-load.md`（讲解、动画、练习都在讲义里）。对应原文：00-overview.md §3；官方教程「Saving and Loading Models」与 `torch.load` 文档（〔在线〕）；本章数字在本机 torch 2.15.0 上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import io
import tempfile

import torch
import torch.nn as nn

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 strict=False | 返回缺了谁 | `strict_false()` | missing_keys=['1.bias'] |
| 第二步 weights_only | 挡住会执行代码的存档 | `weights_only_blocks()` | UnpicklingError |
| 第三步 存档大小 | 参数 4 + AdamW 8 字节 | `checkpoint_sizes()` | 4.0 MB / 8.0 MB |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `size_mb`**（来自 `ch10_save_load.py`）：下面要用到的小工具。

In [ ]:
def size_mb(obj) -> float:
    buf = io.BytesIO()
    torch.save(obj, buf)
    return buf.tell() / 1e6

**函数 `checkpoint_sizes`**（来自 `ch10_save_load.py`）：nn.Linear(n, n) 训一步后，模型和 AdamW 存档各多少 MB。

In [ ]:
def checkpoint_sizes(n: int = 1000):
    """nn.Linear(n, n) 训一步后，模型和 AdamW 存档各多少 MB。"""
    m = nn.Linear(n, n)
    opt = torch.optim.AdamW(m.parameters())
    m(torch.randn(2, n)).sum().backward()
    opt.step()
    return size_mb(m.state_dict()), size_mb(opt.state_dict()), list(opt.state_dict()["state"][0].keys())

**函数 `strict_false`**（来自 `ch10_save_load.py`）：下面要用到的小工具。

In [ ]:
def strict_false():
    m = nn.Sequential(nn.Linear(2, 2), nn.Linear(2, 2))
    sd = dict(m.state_dict())
    sd.pop("1.bias")
    return m.load_state_dict(sd, strict=False)

**类 `Evil`**（来自 `ch10_save_load.py`）：下面要用到的小工具。

In [ ]:
class Evil:
    def __reduce__(self):
        return (print, ("!!! 存档里的代码被执行了",))

**函数 `weights_only_blocks`**（来自 `ch10_save_load.py`）：下面要用到的小工具。

In [ ]:
def weights_only_blocks() -> str:
    with tempfile.NamedTemporaryFile(suffix=".pt") as f:
        torch.save({"x": Evil()}, f.name)
        try:
            torch.load(f.name)
            return "加载成功（危险）"
        except Exception as e:
            return type(e).__name__

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 strict=False 返回：", strict_false())

In [ ]:
print("\n第二步 weights_only 默认挡住了：", weights_only_blocks())

In [ ]:
print("\n第三步 1000×1000 Linear：模型 / AdamW / 状态键 =", checkpoint_sizes())

## 3. 代码里才看得到的

1. **`Evil.__reduce__` 就是攻击方式**：pickle 加载时会调用它返回的函数，所以只加载可信文件。
2. **AdamW 要先 step 一次才有状态**：新建的优化器 `state` 是空的，存档很小。

## 4. 自己改一改

- 把 `checkpoint_sizes(2000)`，看大小是不是变成约 4 倍。
- 在 `checkpoint_sizes` 里换成 SGD（不带 momentum），看优化器存档多大。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 存清单不存对象，先建结构再填数；
> pickle 会执行代码，weights_only 只收纯数据；
> 续训存全套：模型、优化器、调度、步数、随机数，AdamW 存档三倍大。